<a href="https://colab.research.google.com/github/jun-1993-p/rag_example_jun.1993.9/blob/chatbot/7%EA%B0%95_%EC%B1%97%EB%B4%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 설치와 Ollama 서버 실행


In [1]:
import subprocess, time, os

!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama olefile chromadb sentence-transformers

# 코랩 NVIDIA 드라이버 경로를 넣어 줘야 Ollama가 GPU를 잡는다
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + env.get("LD_LIBRARY_PATH", "")
subprocess.run("pkill -f 'ollama serve'", shell=True)
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                 stderr=subprocess.STDOUT, env=env)
time.sleep(8)

!ollama pull gemma4:12b
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!grep -iE "inference compute|no compatible" /content/ollama.log | tail -2

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

# 드라이브 연결과 HWP 업로드

In [2]:
from google.colab import files, drive

drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/rag_rules_cache"

uploaded = files.upload()
HWP_PATH = next(iter(uploaded))
print("업로드:", HWP_PATH)

Mounted at /content/drive


Saving 표준취업규칙(2026년, 배포).hwp to 표준취업규칙(2026년, 배포).hwp
업로드: 표준취업규칙(2026년, 배포).hwp


# 텍스트 추출, 정제, 파트 분리

In [4]:
import re, struct, zlib
import numpy as np
import olefile

EXTENDED = {1, 2, 3, 11, 12, 14, 15, 16, 17, 18, 21, 22, 23}
INLINE = {4, 5, 6, 7, 8, 9, 19, 20}

def decode_para(b: bytes) -> str:
    """문단 텍스트 디코딩. 표·그림 같은 제어문자는 8글자 단위라 통째로 건너뛴다."""
    u = struct.unpack(f"<{len(b)//2}H", b[:len(b)//2*2])
    out, i = [], 0
    while i < len(u):
        c = u[i]
        if c in EXTENDED or c in INLINE:
            if c == 9:
                out.append("\t")
            i += 8
        elif c < 32:
            i += 1
        else:
            out.append(chr(c))
            i += 1
    return "".join(out)

def hwp_to_text(path: str) -> str:
    """표 안의 글자까지 전부 뽑는다. (hwp5txt는 표 내용을 <표>로만 남김)"""
    ole = olefile.OleFileIO(path)
    compressed = ole.openstream("FileHeader").read()[36] & 1
    sections = sorted([s for s in ole.listdir() if s[0] == "BodyText"],
                      key=lambda s: int(s[1][7:]))
    lines = []
    for s in sections:
        data = ole.openstream(s).read()
        if compressed:
            data = zlib.decompress(data, -15)
        i = 0
        while i < len(data):
            header = struct.unpack_from("<I", data, i)[0]
            i += 4
            tag, size = header & 0x3FF, header >> 20
            if size == 0xFFF:
                size = struct.unpack_from("<I", data, i)[0]
                i += 4
            if tag == 67:                      # HWPTAG_PARA_TEXT
                lines.append(decode_para(data[i:i + size]))
            i += size
    return "\n".join(lines)

def clean_text(raw: str) -> str:
    """목차 줄, 작성 가이드([필수]/[선택]/☞/※) 제거"""
    out = []
    for line in raw.splitlines():
        s = line.strip()
        if not s:
            continue
        if re.search(r"\t\s*\d+\s*$", line):
            continue
        if s.startswith(("[필수]", "[선택]", "☞", "※")):
            continue
        out.append(line.rstrip())
    return "\n".join(out)

def split_parts(text: str) -> dict[str, str]:
    """일반 근로자용 / 별첨 괴롭힘 규정 / 단시간 근로자용으로 분리. [별지] 서식은 버린다."""
    parts, current = {}, None
    for line in text.splitlines():
        s = line.strip()
        if s == "일반 근로자용":
            current = "일반 근로자용"
        elif s == "단시간 근로자용":
            current = "단시간 근로자용"
        elif s.startswith("[별첨]"):
            current = "별첨 괴롭힘 규정"
        elif s.startswith("[별지"):
            current = None
        elif current:
            parts.setdefault(current, []).append(line)
    return {k: "\n".join(v) for k, v in parts.items()}

raw = hwp_to_text(HWP_PATH)
clean = clean_text(raw)
parts = split_parts(clean)
print(f"원문 {len(raw):,}자 → 정제 후 {len(clean):,}자")
print("파트:", {k: f"{len(v):,}자" for k, v in parts.items()})

원문 123,407자 → 정제 후 85,493자
파트: {'일반 근로자용': '37,597자', '별첨 괴롭힘 규정': '5,356자', '단시간 근로자용': '35,618자'}


# 청킹 전략 비교

In [5]:
QUICK = True   # True: 비교 전략 2개(빠름) / False: 5개 전부

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """글자 수 기준 고정 청킹"""
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

ARTICLE = re.compile(r"^(제\d+조(?:의\d+)?\s*\([^)]*\))", re.M)
CHAPTER = re.compile(r"^\s*(제\s*\d+\s*장)\s*(.*)$")

def chunk_by_article(text: str, max_len: int = 800) -> list[str]:
    """'제N조(제목)' 단위 청킹. 긴 조항은 제목을 붙여 다시 자른다."""
    pieces = ARTICLE.split(text)
    chunks, seen = [], set()
    for k in range(1, len(pieces), 2):
        title = pieces[k].strip()
        body = (pieces[k] + pieces[k + 1]).strip()
        if len(body) < 20 or body in seen:
            continue
        seen.add(body)
        if len(body) <= max_len:
            chunks.append(body)
        else:
            for p in chunk_text(body, max_len, 100):
                chunks.append(p if p.startswith(title) else f"{title} (계속)\n{p}")
    return chunks

def article_records(text: str, part: str) -> list[dict]:
    """조항 청크 + 메타데이터(파트, 장, 조항명)"""
    records, chapter = [], "기타"
    for block in re.split(r"(?m)^(?=\s*제\s*\d+\s*장\s)", text):
        m = CHAPTER.match(block.split("\n", 1)[0])
        if m:
            chapter = m.group(1).replace(" ", "") + " " + re.sub(r"\s+", " ", m.group(2)).strip()
        for c in chunk_by_article(block):
            records.append({"text": c, "part": part, "chapter": chapter,
                            "article": ARTICLE.match(c).group(1)})
    return records

def noise_tags(chunk: str) -> list[str]:
    """청크에 섞인 노이즈 종류"""
    tags = []
    if len(ARTICLE.findall(chunk)) >= 2:
        tags.append("여러 조항 섞임")
    if re.search(r"\[필수\]|\[선택\]|☞", chunk):
        tags.append("작성 가이드 포함")
    if re.search(r"\t\s*\d+\s*$", chunk, re.M):
        tags.append("목차 포함")
    if not ARTICLE.match(re.sub(r"^\[[^\]]*\]\s*", "", chunk.lstrip())):
        tags.append("조항 중간에서 시작")
    return tags

all_strategies = {
    "fixed_200":   chunk_text(raw, 200, 20),
    "fixed_500":   chunk_text(raw, 500, 50),
    "fixed_1000":  chunk_text(raw, 1000, 100),
    "article_raw": chunk_by_article(raw),
    "article":     chunk_by_article(parts["일반 근로자용"]),
}
for name, chunks in all_strategies.items():
    noisy = sum(1 for c in chunks if noise_tags(c))
    print(f"{name:12s} 청크 {len(chunks):4d}개 / 평균 {int(np.mean([len(c) for c in chunks])):4d}자 / 노이즈 청크 {noisy / len(chunks):.0%}")

keep = ("fixed_500", "article") if QUICK else tuple(all_strategies)
strategies = {k: all_strategies[k] for k in keep}
print("\n임베딩 비교 대상:", list(strategies), f"(총 {sum(map(len, strategies.values()))}개)")

print("\n--- fixed_500 예시 ---")
print(all_strategies["fixed_500"][40])
print("\n--- article 예시 (제33조) ---")
print(next(c for c in all_strategies["article"] if c.startswith("제33조")))

fixed_200    청크  686개 / 평균  199자 / 노이즈 청크 100%
fixed_500    청크  275개 / 평균  498자 / 노이즈 청크 100%
fixed_1000   청크  138개 / 평균  993자 / 노이즈 청크 100%
article_raw  청크  413개 / 평균  302자 / 노이즈 청크 88%
article      청크  108개 / 평균  351자 / 노이즈 청크 0%

임베딩 비교 대상: ['fixed_500', 'article'] (총 383개)

--- fixed_500 예시 ---
로자대표와 서면 합의로 정한 시간을 근로한 것으로 본다. 서면 합의 시 다음 각 호의 사항을 명시하여야 한다.
  1. 대상 업무
  2. 회사가 업무의 수행 수단 및 시간 배분 등에 관하여 사원에게 구체적인 지시를 하지 아니한다는 내용
  3. 근로시간의 산정은 그 서면 합의로 정하는 바에 따른다는 내용
[선택] 업무수행 방법에 재량이 큰 아래 업무에 대하여는 근로자대표와 서면합의로 정한 시간을 근로한 것으로 볼 수 있음

[재량근로의 대상업무]
1. 신상품 또는 신기술의 연구개발이나 인문사회과학 또는 자연과학분야의 연구 업무
2. 정보처리시스템의 설계 또는 분석 업무
3. 신문, 방송 또는 출판 사업에서의 기사의 취재, 편성 또는 편집 업무
4. 의복·실내장식·공업제품·광고 등의 디자인 또는 고안 업무
5. 방송 프로그램·영화 등의 제작 사업에서의 프로듀서나 감독 업무
6. 그 밖에 고용노동부장관이 정하는 업무(회계·법률사건·납세·법무·노무관리·특허·감정평가·금융투자분석·투자자산

--- article 예시 (제33조) ---
제33조(연차유급휴가) ① 1년간 80퍼센트 이상 출근한 사원에게는 15일의 유급휴가를 준다.
  ② 계속하여 근로한 기간이 1년 미만인 사원 또는 1년간 80퍼센트 미만 출근한 사원에게 1개월 개근 시 1일의 유급휴가를 준다.
  ③ 3년 이상 근속한 사원에 대하여는 제1항 규정에 따른 휴가에 최초 1년을

# 임베딩과 벡터DB 저장

In [6]:
import os, hashlib, time
import torch, chromadb
from collections import Counter
from sentence_transformers import SentenceTransformer

os.makedirs(CACHE_DIR, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(CPU)")

embedder = SentenceTransformer("BAAI/bge-m3", device="cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    embedder.half()                  # fp16: 속도↑ 메모리↓
embedder.max_seq_length = 1024       # 청크 최대 800자라 충분

def embed_texts(texts: list[str], batch: int = 32) -> np.ndarray:
    """정규화된 임베딩 → 내적 = 코사인 유사도"""
    return embedder.encode(
        texts, batch_size=batch, normalize_embeddings=True,
        show_progress_bar=len(texts) > 50, convert_to_numpy=True,
    ).astype(np.float32)

def embed_cached(name: str, texts: list[str]) -> np.ndarray:
    """청크 내용이 같으면 드라이브에 저장된 임베딩을 재사용"""
    key = hashlib.md5("\u241e".join(texts).encode()).hexdigest()[:10]
    path = f"{CACHE_DIR}/{name}_{key}.npy"
    if os.path.exists(path):
        print(f"[캐시] {name} ({len(texts)}개)")
        return np.load(path)
    t = time.time()
    vecs = embed_texts(texts)
    np.save(path, vecs)
    print(f"[임베딩] {name} ({len(texts)}개) {time.time() - t:.1f}초")
    return vecs

records = [r for name, t in parts.items() for r in article_records(t, name)]
docs = [f"[{r['part']}] {r['text']}" for r in records]
print(Counter(r["part"] for r in records))

chunk_embeddings = embed_cached("records", docs)

client = chromadb.Client()
if "rules" in [c.name for c in client.list_collections()]:
    client.delete_collection("rules")
collection = client.create_collection("rules", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[f"chunk_{i}" for i in range(len(records))],
    documents=docs,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[{k: r[k] for k in ("part", "chapter", "article")} for r in records],
)
print("저장된 청크:", collection.count())

GPU: Tesla T4


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Counter({'일반 근로자용': 107, '단시간 근로자용': 105, '별첨 괴롭힘 규정': 20})


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

[임베딩] records (232개) 3.1초
저장된 청크: 232


# 벡터 검색

In [7]:
def search_vectordb(query: str, top_k: int = 3, part: str | None = "일반 근로자용"):
    """cosine 공간이므로 유사도 = 1 - distance"""
    q_emb = embed_texts([query])[0].tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=top_k,
                           where={"part": part} if part else None)
    return [
        {"id": i, "text": t, "metadata": m, "similarity": 1 - d}
        for i, t, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

def show_search(query: str, top_k: int = 3, part: str | None = "일반 근로자용"):
    print(f"=== 질문: {query}  (필터: {part}) ===")
    for rank, r in enumerate(search_vectordb(query, top_k, part), 1):
        m = r["metadata"]
        print(f"\n[{rank}위] 유사도 {r['similarity']:.4f} | {m['part']} / {m['chapter']} / {m['article']}")
        print(r["text"][:150] + ("..." if len(r["text"]) > 150 else ""))

show_search("연차 휴가는 며칠이나 되나요?")
print("\n" + "=" * 60 + "\n")
show_search("연차 휴가는 며칠이나 되나요?", part=None)

=== 질문: 연차 휴가는 며칠이나 되나요?  (필터: 일반 근로자용) ===

[1위] 유사도 0.6735 | 일반 근로자용 / 제6장 휴일․휴가 / 제33조(연차유급휴가)
[일반 근로자용] 제33조(연차유급휴가) ① 1년간 80퍼센트 이상 출근한 사원에게는 15일의 유급휴가를 준다.
  ② 계속하여 근로한 기간이 1년 미만인 사원 또는 1년간 80퍼센트 미만 출근한 사원에게 1개월 개근 시 1일의 유급휴가를 준다.
  ③ 3년 이상 근속...

[2위] 유사도 0.6428 | 일반 근로자용 / 제6장 휴일․휴가 / 제34조(연차유급휴가의 사용)
[일반 근로자용] 제34조(연차유급휴가의 사용) ① 연차유급휴가는 사원이 청구한 시기에 주어야 한다. 다만, 청구한 시기에 휴가를 주는 것이 사업 운영에 막대한 지장이 있는 경우 그 시기를 변경할 수 있다.
  ② 사원의 연차유급휴가는 1년간(계속하여 근로한 기간이 1...

[3위] 유사도 0.6021 | 일반 근로자용 / 제4장 인 사 / 제17조(휴직사유 및 기간)
[일반 근로자용] 제17조(휴직사유 및 기간) (계속)
녀(이하 “가족”이라 한다)의 질병, 사고, 노령으로 인하여 그 가족을 돌보기 위하여 필요한 경우(이하 이에 따른 휴직을 “가족돌봄휴직”이라 한다): 연간 90일 이내, 1회 30일 이상
   6. 사원이 ｢공직선...


=== 질문: 연차 휴가는 며칠이나 되나요?  (필터: None) ===

[1위] 유사도 0.6735 | 일반 근로자용 / 제6장 휴일․휴가 / 제33조(연차유급휴가)
[일반 근로자용] 제33조(연차유급휴가) ① 1년간 80퍼센트 이상 출근한 사원에게는 15일의 유급휴가를 준다.
  ② 계속하여 근로한 기간이 1년 미만인 사원 또는 1년간 80퍼센트 미만 출근한 사원에게 1개월 개근 시 1일의 유급휴가를 준다.
  ③ 3년 이상 근속...

[2위] 유사도 0.6612 | 단시간 근로자용 / 제6장 휴일・휴가 / 제32조(연차휴가의 사용)
[단시간 근로자용] 제32조(연차휴

# 청킹 전략별 검색 결과 비교

In [8]:
strategy_vecs = {name: embed_cached(name, chunks) for name, chunks in strategies.items()}

def compare_chunking(query: str, top_k: int = 3):
    print(f"=== 질문: {query} ===")
    q = embed_texts([query])[0]
    for name, chunks in strategies.items():
        scores = strategy_vecs[name] @ q
        print(f"\n##### {name}")
        for rank, i in enumerate(np.argsort(scores)[::-1][:top_k], 1):
            c = chunks[i]
            print(f"[{rank}위] {scores[i]:.4f}  노이즈: {', '.join(noise_tags(c)) or '없음'}")
            print("      " + re.sub(r"\s+", " ", c)[:100] + "...")

compare_chunking("연차 휴가는 며칠이나 되나요?")

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

[임베딩] fixed_500 (275개) 3.8초


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[임베딩] article (108개) 1.2초
=== 질문: 연차 휴가는 며칠이나 되나요? ===

##### fixed_500
[1위] 0.6579  노이즈: 여러 조항 섞임, 작성 가이드 포함, 조항 중간에서 시작
       기준으로 재산정한다는 별도의 단서가 없는 한 발생한 휴가 일수 전체를 부여해야 함(임금근로시간정책팀-489, 2008.2.28. 등) 제32조(연차휴가의 사용) ① 연차유급휴가는...
[2위] 0.6539  노이즈: 작성 가이드 포함, 조항 중간에서 시작
       2021다227100, 임금근로시간과-2861, 2021.12.15.) ☞ (참고) 1년 미만의 ’1개월‘ 단위 연차유급휴가는 회계연도 기준으로 부여할 수 없음(2018.5월, ...
[3위] 0.6495  노이즈: 작성 가이드 포함, 조항 중간에서 시작
      ) ☞ (참고) 1년 미만의 ’1개월‘ 단위 연차유급휴가는 회계연도 기준으로 부여할 수 없음(2018.5월, 개정 근로기준법 설명자료) ☞ (참고) 2018.5.29. 이후 개시하...

##### article
[1위] 0.6742  노이즈: 없음
      제33조(연차유급휴가) ① 1년간 80퍼센트 이상 출근한 사원에게는 15일의 유급휴가를 준다. ② 계속하여 근로한 기간이 1년 미만인 사원 또는 1년간 80퍼센트 미만 출근한 사원...
[2위] 0.6339  노이즈: 없음
      제34조(연차유급휴가의 사용) ① 연차유급휴가는 사원이 청구한 시기에 주어야 한다. 다만, 청구한 시기에 휴가를 주는 것이 사업 운영에 막대한 지장이 있는 경우 그 시기를 변경할 ...
[3위] 0.5970  노이즈: 없음
      제35조(연차유급휴가의 사용촉진) ① 회사는 제33조 제1항·제2항 및 제3항에 따른 연차유급휴가(계속하여 근로한 기간이 1년 미만인 사원에게 발생한 유급휴가는 제외)의 사용을 촉...


# 임베딩 3D 시각화

In [9]:
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go

def plot_embeddings_3d(query: str | None = None, part: str = "일반 근로자용", top_k: int = 5):
    idx = [i for i, r in enumerate(records) if r["part"] == part]
    pca = PCA(n_components=3)
    vis = pca.fit_transform(chunk_embeddings[idx])
    ratio = pca.explained_variance_ratio_.sum()

    df = {
        "x": vis[:, 0], "y": vis[:, 1], "z": vis[:, 2],
        "chapter": [records[i]["chapter"] for i in idx],
        "article": [records[i]["article"] for i in idx],
        "preview": [re.sub(r"\s+", " ", records[i]["text"])[:60] for i in idx],
    }
    fig = px.scatter_3d(
        df, x="x", y="y", z="z", color="chapter",
        hover_name="article", hover_data={"preview": True, "x": False, "y": False, "z": False},
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        title=f"취업규칙 조항 임베딩 3D ({part}, PCA 설명력 {ratio:.1%})",
    )
    fig.update_traces(marker=dict(size=5, opacity=0.8))

    if query:
        q = pca.transform(embed_texts([f"[{part}] {query}"]))[0]
        hits = search_vectordb(query, top_k=top_k, part=part)
        pos = [idx.index(int(h["id"].split("_")[1])) for h in hits]
        fig.add_trace(go.Scatter3d(
            x=vis[pos, 0], y=vis[pos, 1], z=vis[pos, 2], mode="markers+text",
            name=f"검색 top-{top_k}", text=[f"{n}위" for n in range(1, len(pos) + 1)],
            hovertext=[h["metadata"]["article"] for h in hits],
            marker=dict(size=10, color="rgba(0,0,0,0)", line=dict(color="black", width=4)),
        ))
        fig.add_trace(go.Scatter3d(
            x=[q[0]], y=[q[1]], z=[q[2]], mode="markers+text", name="질문",
            text=["Q"], hovertext=[query], marker=dict(size=12, color="red", symbol="diamond"),
        ))
        for p in pos:
            fig.add_trace(go.Scatter3d(
                x=[q[0], vis[p, 0]], y=[q[1], vis[p, 1]], z=[q[2], vis[p, 2]], mode="lines",
                line=dict(color="red", width=2, dash="dash"), showlegend=False, hoverinfo="skip",
            ))

    fig.update_layout(height=750, legend=dict(itemsizing="constant"))
    fig.show()

plot_embeddings_3d("연차 휴가는 며칠이나 되나요?")


# 청킹 전략별 임베딩 분포

In [10]:
def plot_chunking_compare():
    names = list(strategy_vecs)
    pca = PCA(n_components=3).fit(np.vstack([strategy_vecs[n] for n in names]))

    fig = go.Figure()
    for n in names:
        vis = pca.transform(strategy_vecs[n])
        hover = [re.sub(r"\s+", " ", c)[:50] + "<br>노이즈: " + (", ".join(noise_tags(c)) or "없음")
                 for c in strategies[n]]
        fig.add_trace(go.Scatter3d(
            x=vis[:, 0], y=vis[:, 1], z=vis[:, 2], mode="markers",
            name=f"{n} ({len(strategies[n])}개)", hovertext=hover, hoverinfo="text",
            marker=dict(size=3, opacity=0.6),
        ))
    fig.update_layout(title="청킹 전략별 임베딩 분포 비교", height=750)
    fig.show()

plot_chunking_compare()

# 챗봇 (추론 후 답변)

In [ ]:
import ollama

CHAT_MODEL = "gemma4:12b"
chat_client = ollama.Client(timeout=300)   # 서버가 멈춰도 무한 대기하지 않게

SYSTEM = ("당신은 회사 취업규칙 안내 챗봇입니다. 반드시 아래 [규정] 내용만 근거로 답하고, "
          "답변 끝에 근거 조항 번호(예: 제33조)를 적으세요. "
          "규정에 없는 내용이면 '취업규칙에서 확인되지 않습니다. 인사팀에 문의하세요.'라고 답하세요.")

def answer(question: str, top_k: int = 4, part: str = "일반 근로자용", show_context: bool = True) -> str:
    hits = search_vectordb(question, top_k, part)
    context = "\n\n---\n\n".join(h["text"] for h in hits)
    if show_context:
        print("[검색된 조항] " + ", ".join(h["metadata"]["article"] for h in hits))
    t = time.time()
    res = chat_client.chat(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"[규정]\n{context}\n\n[질문]\n{question}"},
        ],
        options={"num_ctx": 8192, "temperature": 0.2, "num_predict": 400},   # 답변 길이 제한
        keep_alive="30m",                                                    # 모델을 메모리에 유지
    )
    if show_context:
        print(f"[생성 {time.time() - t:.1f}초]")
    return res["message"]["content"]

for q in ["나 입사 8개월 됐는데 연차 며칠이야?",
          "연차 쓰려면 어떻게 해야 돼?",
          "결혼하면 휴가 며칠 줘?",
          "회사 주차 지원금 있어?"]:
    print(f"\nQ. {q}")
    print(f"A. {answer(q)}")

!ollama ps

# 직접 질문하려면 아래 주석을 풀고 실행
# while (q := input("\n질문 (엔터 = 종료): ").strip()):
#     print(answer(q))



Q. 나 입사 8개월 됐는데 연차 며칠이야?
[검색된 조항] 제33조(연차유급휴가), 제61조 (퇴직급여제도의 설정), 제29조(연장․야간 및 휴일근로), 제55조(상여금 지급)
[생성 225.0초]
A. 

Q. 연차 쓰려면 어떻게 해야 돼?
[검색된 조항] 제35조(연차유급휴가의 사용촉진), 제34조(연차유급휴가의 사용), 제35조(연차유급휴가의 사용촉진), 제33조(연차유급휴가)


# 챗봇 (생각 과정 생략)

In [ ]:
import ollama

CHAT_MODEL = "gemma4:12b"
THINK = False                              # True면 추론 후 답변 (느리지만 계산 문제에 유리)
chat_client = ollama.Client(timeout=300)

SYSTEM = ("당신은 회사 취업규칙 안내 챗봇입니다. 반드시 아래 [규정] 내용만 근거로 답하고, "
          "답변 끝에 근거 조항 번호(예: 제33조)를 적으세요. "
          "규정에 없는 내용이면 '취업규칙에서 확인되지 않습니다. 인사팀에 문의하세요.'라고 답하세요.")

def answer(question: str, top_k: int = 4, part: str = "일반 근로자용",
           show_context: bool = True, think: bool = THINK) -> str:
    hits = search_vectordb(question, top_k, part)
    context = "\n\n---\n\n".join(h["text"] for h in hits)
    if show_context:
        print("[검색된 조항] " + ", ".join(h["metadata"]["article"] for h in hits))
    t = time.time()
    res = chat_client.chat(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"[규정]\n{context}\n\n[질문]\n{question}"},
        ],
        think=think,
        options={"num_ctx": 8192, "temperature": 0.2,
                 "num_predict": 2048 if think else 600},
        keep_alive="30m",
    )
    msg = res["message"]
    if show_context:
        n_think = len(msg.get("thinking") or "")
        print(f"[생성 {time.time() - t:.1f}초 | 추론 {n_think}자]")
    content = msg["content"].strip()
    if not content:
        content = "(답변이 비었습니다. 추론에 토큰을 다 썼을 수 있습니다. num_predict를 늘려 보세요.)"
    return content

for q in ["나 입사 8개월 됐는데 연차 며칠이야?",
          "연차 쓰려면 어떻게 해야 돼?",
          "결혼하면 휴가 며칠 줘?",
          "회사 주차 지원금 있어?"]:
    print(f"\nQ. {q}")
    print(f"A. {answer(q)}")

!ollama ps

# 직접 질문하려면 아래 주석을 풀고 실행
# while (q := input("\n질문 (엔터 = 종료): ").strip()):
#     print(answer(q))